# Part 2 — Group Presentation & Notebook

**Group members:** 

- Dargys Perez : `Data Engineer`
  
- Cecilia Back : `Data Analyst`

- Marta Bielow : `Accounting Manager`
  
- Jessika Rosen 

## Compliance with the 10-day rule (delivery to invoice) was the focus of the work meeting.

**The following issues were raised:**
- Why does the delivery_confirmed_date in Deliveries.csv (originating from the logistics system/TMS) contain multiple date formats?
- What is the current situation regarding the time gap between delivery_confirmed_date and invoice_date? 
- Based on available historical data, how well are we complying with the 10-day rule today?
 

In [24]:
# using pandas for data analysis
import pandas as pd

In [25]:
# load the relevant data for the 10 rules analysis
invoice_line = pd.read_csv("invoice_line.csv")
deliveries = pd.read_csv("deliveries.csv")

In [ ]:
invoice_line['invoice_date']

In [ ]:
deliveries['delivery_confirm_date']

In [ ]:
deliveries.dtypes

In [ ]:
invoice_line.dtypes

In [ ]:
invoice_line.groupby("order_id")["invoice_date"].nunique().value_counts()

In [ ]:
invoice_line.groupby("order_id")["invoice_date"].value_counts()

In [26]:
#Normalize invoice_line data to one row per order_id
invoice_order = (
    invoice_line
    .drop_duplicates(subset=["order_id"])
    [["order_id", "invoice_date"]]
)

In [ ]:
# Ensure that the data is cleaned from duplicates.
invoice_order.value_counts()

In [27]:
#Join datasets
df = invoice_order.merge(
    deliveries,
    on="order_id",
    how="left"
)

In [28]:
#Convert to datetime
df["invoice_date"] = pd.to_datetime(df["invoice_date"])


In [29]:
# the delivery_confirm_date has data quality issues, the date is given in differents formats and I need first to normalize.
df["delivery_confirm_date"] = pd.to_datetime(
    df["delivery_confirm_date"],
    format="mixed",
    dayfirst=True,
    errors="coerce"
)

In [30]:
#Calculate the date difference
df["days_diff"] = (df["invoice_date"] - df["delivery_confirm_date"]).dt.days

#Categorize the result
df["status"] = "Fulfill 10-day-rule"
df.loc[df["days_diff"] > 10, "status"] = "Late"
df.loc[df["days_diff"] < 0, "status"] = "Negative date-diff"
df.loc[df["days_diff"].isna(), "status"] = "Missing"

df["status"].value_counts()

status
Fulfill 10-day-rule    22
Late                   16
Negative date-diff     12
Name: count, dtype: int64

In [ ]:
# How the "days_diff" are distributed in the dataset.
df["days_diff"].value_counts().sort_index()

days_diff
-261    1
-259    1
-227    2
-176    1
-166    1
-143    1
-139    1
-71     1
-67     1
-54     1
-9      1
 4      1
 5      3
 6      6
 7      1
 8      6
 9      1
 10     4
 13     1
 14     1
 15     2
 16     1
 19     1
 20     2
 22     2
 23     1
 25     1
 27     2
 40     1
 76     1
Name: count, dtype: int64

In [ ]:
# Sorting out the orders with negative day_diff.
df[df["status"] == "Negative date-diff"]

,order_id,invoice_date,delivery_confirm_id,delivery_confirm_date,delivery_country,days_diff,status
6,ORD-2026-483219,2027-02-06,CONF-560965,2027-08-01,Netherlands,-176,Negative date-diff
9,ORD-2027-296460,2027-02-17,CONF-747653,2027-10-02,France,-227,Negative date-diff
10,ORD-2027-340184,2027-02-21,CONF-831485,2027-03-02,Sweden,-9,Negative date-diff
12,ORD-2027-881543,2027-03-24,CONF-699281,2027-06-03,Germany,-71,Negative date-diff
15,ORD-2027-126344,2027-03-17,CONF-381172,2027-12-03,France,-261,Negative date-diff
16,ORD-2026-339139,2027-02-24,CONF-871248,2027-05-02,France,-67,Negative date-diff
26,ORD-2027-966069,2027-02-17,CONF-941444,2027-10-02,Germany,-227,Negative date-diff
30,ORD-2027-599775,2027-03-17,CONF-260684,2027-08-03,Belgium,-139,Negative date-diff
36,ORD-2027-116776,2027-03-10,CONF-198879,2027-05-03,Netherlands,-54,Negative date-diff
42,ORD-2027-346967,2027-02-17,CONF-526176,2027-08-02,Netherlands,-166,Negative date-diff


## Data Quality Findings – Delivery vs Invoice Dates

An analysis of the time difference between delivery_confirmed_date and invoice_date identified a number of cases with negative values, indicating that the invoice date precedes the delivery confirmation date.

**A subset of these cases was investigated manually in the source CSV files:**.<br>

*- Several records were confirmed to be data quality issues, primarily caused by:*.<br>
- Inconsistent or incorrect date formats.<br>
- Incorrectly recorded delivery dates.<br>
  
*- Other records could not be conclusively explained based on available data. Possible explanations include:* <br>

- Delayed registration of delivery confirmations in the logistics system (TMS).<br>
- Differences in business process (e.g. invoicing prior to delivery).<br>
- Integration or timing discrepancies between systems.<br>

*Further clarification would require:*
- Validation of business rules for invoicing vs delivery.<br>
- Review of source system definitions and event timestamps.<br>